# Hierarchical Graph Representation with DiffPool on PROTEINS

Graph Classification on PROTEINS (TUDataset): DiffPool ("Hierarchical Graph Representation Learning with Differentiable Pooling", https://arxiv.org/abs/1806.08804) learns a soft cluster-assignment matrix at each level and coarsens the graph twice (150 → 38 → 10 nodes), using one GNN to produce the assignment and a separate GNN to produce the embedding at each level. This notebook ports the reference implementation to K3-Node: `k3_node.transforms.ToDense` pre-pads every graph (dropping any graph over 150 nodes) into fixed-size `(x, adj, mask)` tensors so `k3_node.loader.DenseDataLoader` can simply stack them, `K3GNN` (three `DenseSAGEConv` + `BatchNormalization` layers, concatenated) is the shared building block for both the pooling and embedding GNNs, and `k3_layers.dense_diff_pool` performs the actual coarsening. Training reports the best test accuracy at the epoch with the best validation accuracy, exactly as in the reference — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

from math import ceil
import numpy as np
import keras
from keras import layers, ops

from k3_node import layers as k3_layers
from k3_node import transforms as k3_transforms
from k3_node.datasets import TUDataset
from k3_node.loader import DenseDataLoader

title = "Hierarchical Graph Representation with DiffPool on PROTEINS"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

max_nodes = 150

# 1. Dataset: pre-pad every graph (<=150 nodes) to fixed-size dense tensors,
# then shuffle and split into test/val/train (10%/10%/80%)
dataset = TUDataset(
    root="./data/PROTEINS_dense",
    name="PROTEINS",
    transform=k3_transforms.ToDense(max_nodes),
    pre_filter=lambda data: data.num_nodes <= max_nodes,
)
perm = np.random.permutation(len(dataset)).tolist()
dataset = dataset[perm]

n = (len(dataset) + 9) // 10
test_dataset = dataset[:n]
val_dataset = dataset[n:2 * n]
train_dataset = dataset[2 * n:]
test_loader = DenseDataLoader(test_dataset, batch_size=20)
val_loader = DenseDataLoader(val_dataset, batch_size=20)
train_loader = DenseDataLoader(train_dataset, batch_size=20)


# 2. Shared building block: 3 x (DenseSAGEConv -> BatchNorm), concatenated
class K3GNN(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels, normalize=False, add_lin=True):
        super().__init__()
        self.conv1 = k3_layers.DenseSAGEConv(in_channels, hidden_channels, normalize)
        self.bn1 = layers.BatchNormalization()
        self.conv2 = k3_layers.DenseSAGEConv(hidden_channels, hidden_channels, normalize)
        self.bn2 = layers.BatchNormalization()
        self.conv3 = k3_layers.DenseSAGEConv(hidden_channels, out_channels, normalize)
        self.bn3 = layers.BatchNormalization()

        self.lin = layers.Dense(out_channels) if add_lin else None

    def bn(self, i, x, training):
        batch_size, num_nodes, num_channels = ops.shape(x)[0], ops.shape(x)[1], ops.shape(x)[2]
        x = ops.reshape(x, (-1, num_channels))
        x = getattr(self, f"bn{i}")(x, training=training)
        return ops.reshape(x, (batch_size, num_nodes, num_channels))

    def call(self, x, adj, mask=None, training=False):
        x1 = self.bn(1, ops.relu(self.conv1(x, adj, mask)), training)
        x2 = self.bn(2, ops.relu(self.conv2(x1, adj, mask)), training)
        x3 = self.bn(3, ops.relu(self.conv3(x2, adj, mask)), training)

        x = ops.concatenate([x1, x2, x3], axis=-1)

        if self.lin is not None:
            x = ops.relu(self.lin(x))

        return x


# 3. Two-level DiffPool network
class K3Net(keras.Model):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        num_nodes = ceil(0.25 * max_nodes)
        self.gnn1_pool = K3GNN(in_channels, 64, num_nodes)
        self.gnn1_embed = K3GNN(in_channels, 64, 64, add_lin=False)

        num_nodes = ceil(0.25 * num_nodes)
        self.gnn2_pool = K3GNN(3 * 64, 64, num_nodes)
        self.gnn2_embed = K3GNN(3 * 64, 64, 64, add_lin=False)

        self.gnn3_embed = K3GNN(3 * 64, 64, 64, add_lin=False)

        self.lin1 = layers.Dense(64)
        self.lin2 = layers.Dense(num_classes)

    def call(self, x, adj, mask=None, training=False):
        s = self.gnn1_pool(x, adj, mask, training=training)
        x = self.gnn1_embed(x, adj, mask, training=training)

        x, adj, l1, e1 = k3_layers.dense_diff_pool(x, adj, s, mask)

        s = self.gnn2_pool(x, adj, training=training)
        x = self.gnn2_embed(x, adj, training=training)

        x, adj, l2, e2 = k3_layers.dense_diff_pool(x, adj, s)

        x = self.gnn3_embed(x, adj, training=training)

        x = ops.mean(x, axis=1)
        x = ops.relu(self.lin1(x))
        x = self.lin2(x)
        # `l1 + l2` / `e1 + e2` are computed to match the reference exactly,
        # but (as in the reference) are not actually added into the loss.
        return ops.log_softmax(x, axis=-1), l1 + l2, e1 + e2


model = K3Net(dataset.num_features, dataset.num_classes)

# Eager forward pass to build every sublayer's weights
sample = next(iter(train_loader))
_ = model(
    ops.convert_to_tensor(sample.x, dtype="float32"),
    ops.convert_to_tensor(sample.adj, dtype="float32"),
    ops.convert_to_tensor(sample.mask, dtype="bool"),
    training=False,
)

optimizer = keras.optimizers.Adam(learning_rate=0.001)
trainable_vars = model.trainable_variables

# Setup optimizer variables for the functional (JAX) backend
if backend == "jax":
    import jax
    optimizer.build(trainable_vars)
    opt_vars = [v.value for v in optimizer.variables]
    non_trainable_vars = [v.value for v in model.non_trainable_variables]


def batch_tensors(data_batch):
    x = ops.convert_to_tensor(data_batch.x, dtype="float32")
    adj = ops.convert_to_tensor(data_batch.adj, dtype="float32")
    mask = ops.convert_to_tensor(data_batch.mask, dtype="bool")
    y = ops.reshape(ops.convert_to_tensor(data_batch.y, dtype="int32"), (-1,))
    return x, adj, mask, y


def nll_loss(log_probs, y, num_classes):
    y_one_hot = ops.one_hot(y, num_classes)
    return -ops.mean(ops.sum(y_one_hot * log_probs, axis=-1))


# 4. Multi-Backend Training Step
def train_step(data_batch):
    global opt_vars, non_trainable_vars
    x, adj, mask, y = batch_tensors(data_batch)
    num_classes = dataset.num_classes
    num_graphs = int(ops.shape(y)[0])

    if backend == "torch":
        out, _, _ = model(x, adj, mask, training=True)
        loss = nll_loss(out, y, num_classes)
        loss.backward()
        grads = [v.value.grad for v in trainable_vars]
        optimizer.apply_gradients(zip(grads, trainable_vars))
        for v in trainable_vars:
            if v.value.grad is not None:
                v.value.grad.zero_()
        loss_value = float(ops.convert_to_numpy(loss))

    elif backend == "tensorflow":
        import tensorflow as tf
        with tf.GradientTape() as tape:
            out, _, _ = model(x, adj, mask, training=True)
            loss = nll_loss(out, y, num_classes)
        grads = tape.gradient(loss, trainable_vars)
        optimizer.apply_gradients(zip(grads, trainable_vars))
        loss_value = float(ops.convert_to_numpy(loss))

    else:  # jax
        trainable_values = [v.value for v in trainable_vars]

        def loss_fn(params):
            (out, _, _), new_non_trainable = model.stateless_call(
                params, non_trainable_vars, x, adj, mask, training=True
            )
            return nll_loss(out, y, num_classes), new_non_trainable

        (loss_val, new_non_trainable), grads = jax.value_and_grad(loss_fn, has_aux=True)(trainable_values)
        new_values, opt_vars = optimizer.stateless_apply(opt_vars, grads, trainable_values)
        for v, val in zip(trainable_vars, new_values):
            v.assign(val)
        for v, val in zip(model.non_trainable_variables, new_non_trainable):
            v.assign(val)
        non_trainable_vars = new_non_trainable
        loss_value = float(loss_val)

    return loss_value * num_graphs


def train():
    total_loss = 0.0
    for data_batch in train_loader:
        total_loss += train_step(data_batch)
    return total_loss / len(train_loader.dataset)


def test(loader):
    correct = 0
    for data_batch in loader:
        x, adj, mask, y = batch_tensors(data_batch)
        out, _, _ = model(x, adj, mask, training=False)
        pred = ops.argmax(out, axis=-1)
        correct += int(ops.convert_to_numpy(ops.sum(ops.cast(pred == ops.cast(y, pred.dtype), "int32"))))
    return correct / len(loader.dataset)


print(f"Training K3-Node DiffPool model on {backend} backend...")
best_val_acc = test_acc = 0.0
for epoch in range(1, 151):
    train_loss = train()
    val_acc = test(val_loader)
    if val_acc > best_val_acc:
        test_acc = test(test_loader)
        best_val_acc = val_acc
    print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, "
          f"Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}")

print("\n✓ K3-Node execution completed successfully!")